In [2]:
import pandas as pd

df_paysim = pd.read_csv("../data/paysim.csv")

print("Dataset shape:", df_paysim.shape)
df_paysim.head()

Dataset shape: (6362620, 11)


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


## 2. Fraud Distribution and Transaction Types

Before sampling, we identify which transaction types contain fraud.
This defines the PaySim-specific modeling scope.

In [3]:
print("Fraud / non-fraud distribution:")
print(df_paysim["isFraud"].value_counts())

fraud_type_counts = (
    df_paysim.loc[df_paysim["isFraud"] == 1, "type"]
    .value_counts()
)

print("\nFraud transaction types:")
print(fraud_type_counts)

fraud_types = fraud_type_counts.index.tolist()

print("\nSelected fraud-related types:")
print(fraud_types)

Fraud / non-fraud distribution:
isFraud
0    6354407
1       8213
Name: count, dtype: int64

Fraud transaction types:
type
CASH_OUT    4116
TRANSFER    4097
Name: count, dtype: int64

Selected fraud-related types:
['CASH_OUT', 'TRANSFER']


## 3. Define the Modeling Scope

Fraud in PaySim appears only in `CASH_OUT` and `TRANSFER`.
The modeling pool is therefore restricted to these transaction types.

This is a PaySim-specific preprocessing decision, not a general rule
for other transaction datasets.

In [4]:
df_scope = df_paysim[
    df_paysim["type"].isin(fraud_types)
].copy()

print("Scoped dataset shape:", df_scope.shape)

print("\nFraud distribution:")
print(df_scope["isFraud"].value_counts())

print("\nTransaction type distribution:")
print(df_scope["type"].value_counts())

print("\nType × fraud:")
print(pd.crosstab(df_scope["type"], df_scope["isFraud"]))

Scoped dataset shape: (2770409, 11)

Fraud distribution:
isFraud
0    2762196
1       8213
Name: count, dtype: int64

Transaction type distribution:
type
CASH_OUT    2237500
TRANSFER     532909
Name: count, dtype: int64

Type × fraud:
isFraud         0     1
type                   
CASH_OUT  2233384  4116
TRANSFER   528812  4097


## 4. Create a Controlled 200k Modeling Sample

All fraud records are preserved.
Non-fraud records are reproducibly sampled from the scoped transaction pool.

In [5]:
df_fraud = df_scope[
    df_scope["isFraud"] == 1
].copy()

df_nonfraud_pool = df_scope[
    df_scope["isFraud"] == 0
].copy()

target_total = 200_000
needed_nonfraud = target_total - len(df_fraud)

print("Fraud records:", len(df_fraud))
print("Non-fraud pool:", len(df_nonfraud_pool))
print("Non-fraud records to sample:", needed_nonfraud)

Fraud records: 8213
Non-fraud pool: 2762196
Non-fraud records to sample: 191787


In [6]:
df_nonfraud = df_nonfraud_pool.sample(
    n=needed_nonfraud,
    random_state=42
)

df_model = pd.concat(
    [df_fraud, df_nonfraud],
    ignore_index=True
)

df_model = (
    df_model
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)

print("Final modeling dataset:", df_model.shape)

Final modeling dataset: (200000, 11)


## 5. Validate and Save the Modeling Dataset

In [7]:
print("Fraud / non-fraud distribution:")
print(df_model["isFraud"].value_counts())

print("\nFraud ratio:")
print(f"{df_model['isFraud'].mean() * 100:.2f}%")

print("\nType × fraud:")
print(pd.crosstab(df_model["type"], df_model["isFraud"]))

print("\nDuplicate rows:", df_model.duplicated().sum())
print("Missing values:", df_model.isna().sum().sum())

print("\nUnique senders:", df_model["nameOrig"].nunique())
print("Unique receivers:", df_model["nameDest"].nunique())

Fraud / non-fraud distribution:
isFraud
0    191787
1      8213
Name: count, dtype: int64

Fraud ratio:
4.11%

Type × fraud:
isFraud        0     1
type                  
CASH_OUT  155046  4116
TRANSFER   36741  4097

Duplicate rows: 0
Missing values: 0

Unique senders: 199991
Unique receivers: 147620


In [9]:
df_model.to_csv(
    "../data/paysim_model_200k.csv",
    index=False
)

In [10]:
from pathlib import Path

output_path = Path("../data/paysim_model_200k.csv")

print("Dosya var mı?:", output_path.exists())
print("Kaydedilen adres:", output_path.resolve())

Dosya var mı?: True
Kaydedilen adres: C:\anaconda13\Scripts\PAYSIM\data\paysim_model_200k.csv
